This is the cleaned-up code in which the remaining cases of Carlitz-extensivity are solved. <br> <br>

Only the deterministic method is included. When the code is run, progress is printed only for every 30'000th cyclotomic coset.

In [ ]:
from sage.all import *

# ============================================================
# Field construction
# ============================================================

def make_extension_field(q, n, gen_name='a'):
    """
    Construct F_{q^n} as an extension of F_q of degree n.
    Returns (F_q, F_q_n) as genuine FiniteField objects with
    F_q installed as a subfield of F_q_n, so that coercion and
    F_q_n.vector_space(F_q, ...) both work.
    """
    from sage.arith.misc import is_prime_power
    p, k = is_prime_power(q, get_data=True)
    if k == 0:
        raise ValueError(f"q = {q} is not a prime power")

    F_q_n = GF(p**(k * n), gen_name)
    if k == 1:
        F_q = F_q_n.prime_subfield()
    else:
        F_q = F_q_n.subfield(k, 'b')

    assert F_q_n.has_coerce_map_from(F_q)
    return F_q, F_q_n


# ============================================================
# q-cyclotomic cosets mod N = q^n - 1
# ============================================================

def nonzero_q_cyclotomic_coset_data(F_q_n, q):
    """
    Yield one record for each q-cyclotomic coset in F_q_n^*.

    If g is a multiplicative generator of F_q_n^* and N = |F_q_n^*|, then the
    nonzero element z = g^e has Frobenius orbit
        z, z^q, z^(q^2), ...
    corresponding exactly to the q-cyclotomic coset of e mod N.

    Yields dictionaries with keys:
        'rep_exp'   : representative exponent e
        'coset_exp' : full exponent coset
        'z0'        : g^e
        'orbit_z'   : [z0, z0^q, z0^(q^2), ...]
    """
    g = F_q_n.multiplicative_generator()
    N = ZZ(F_q_n.order() - 1)

    seen = bytearray(int(N))

    for e in range(int(N)):
        if seen[e]:
            continue

        x = e
        z0 = g**e
        w = z0
    
        coset = []
        orbit_z = []

        while not seen[x]:
            seen[x] = 1

            coset.append(ZZ(x))
            orbit_z.append(w)

            x = (x * q) % N
            w = w**q
            

        yield {
            'rep_exp': ZZ(e),
            'coset_exp': coset,
            'z0': z0,
            'orbit_z': orbit_z,
        }


# ============================================================
# Minimal polynomial over F_q from Frobenius orbit
# ============================================================

def minpoly_from_q_frobenius_orbit(orbit_z, F_q, var='t'):
    """
    Given the q-Frobenius orbit [z, z^q, z^(q^2), ...], return the minimal
    polynomial of z over F_q.

    Output:
        f_z in F_q[var]
    """
    F_q_polynomials = PolynomialRing(F_q, var)

    F_q_n = orbit_z[0].parent()
    F_q_n_polynomials = PolynomialRing(F_q_n, 'U')
    U = F_q_n_polynomials.gen()

    f_over_extension = prod(U - w for w in orbit_z)
    f_z = F_q_polynomials([F_q(c) for c in f_over_extension.list()])
    return f_z


# ============================================================
# Primitive projective-point candidates in P(F_q_n)
# ============================================================

def primitive_point_pair_iter(F_q, F_q_n, to_V):
    """
    Iterate over primitive projective F_q-points of F_q_n.

    For each such point, yield the canonical exponent (the unique
    representative in {1, ..., Q-1}), the corresponding field element
    g^k, and its image as a vector in (F_q)^n.

    Let N = q^n - 1 and Q = (q^n - 1)/(q - 1), and let g be a
    multiplicative generator of F_q_n^*.

    From the theory:
      - Any projective point [g^k] has a unique representative g^l
        with 0 <= l <= Q-1.
      - The point [g^k] is primitive iff gcd(k, Q) = 1.
      - [g^0] = [1] corresponds to F_q^* itself, which is not primitive
        and is therefore skipped.

    Yields triples:
        (point_exp, x_rep, v_rep)
    where:
        point_exp is k in {1, ..., Q-1} with gcd(k, Q) = 1,
        x_rep     = g^k is the representative of that primitive point,
        v_rep     = to_V(x_rep) is the corresponding vector in (F_q)^n.
    """
    g = F_q_n.multiplicative_generator()
    N = ZZ(F_q_n.order() - 1)
    q = F_q.order()
    Q = N // (q - 1)

    x = g  # x = g^1
    for k in range(1, int(Q)):
        if gcd(k, Q) == 1:
            yield (ZZ(k), x, to_V(x))
        x *= g


def primitive_point_pair_list(F_q, F_q_n, to_V):
    """
    Materialize primitive-point representatives once.
    """
    return list(primitive_point_pair_iter(F_q, F_q_n, to_V))


def primitive_multiple_on_same_point(F_q, F_q_n, point_exp, x_rep):
    """
    Given a representative x_rep = g^k of a primitive F_q-point, find a scalar
    multiple lambda*x_rep that is actually primitive in F_q_n^*.

    We search among:
        x_rep, h*x_rep, h^2*x_rep, ..., h^(q-2)*x_rep
    where h = g^Q generates F_q^* and Q = (q^n - 1)/(q - 1).

    Returns:
        primitive_x, primitive_exp
    """
    g = F_q_n.multiplicative_generator()
    N = ZZ(F_q_n.order() - 1)
    q = F_q.order()
    Q = N // (q - 1)
    h = g**Q  # generator of F_q^*

    y = x_rep
    a = ZZ(point_exp) % N

    for _ in range(q - 1):
        if gcd(a, N) == 1:
            return y, a
        y *= h
        a = (a + Q) % N

    raise RuntimeError(
        "No primitive scalar multiple found on a primitive point; this should not happen."
    )


# ============================================================
# Linear algebra over F_q
# ============================================================

def prepare_linear_algebra(F_q, F_q_n):
    """
    Prepare F_q_n as an F_q-vector space, together with conversion maps.

    Returns:
        V            : F_q-vector space
        from_V       : V -> F_q_n
        to_V         : F_q_n -> V
        basis_F_q_n  : basis of F_q_n over F_q, as field elements
        zero_v       : zero vector in V
    """
    V, from_V, to_V = F_q_n.free_module(base=F_q, map=True)
    basis_F_q_n = [from_V(v) for v in V.basis()]
    zero_v = V.zero()
    return V, from_V, to_V, basis_F_q_n, zero_v


def gamma_matrix_row_action(z, q, F_q, basis_F_q_n, to_V):
    """
    Matrix A over F_q for gamma_z(y) = z*y + y^q, in row-action convention:
        to_V(x) * A = to_V(gamma_z(x)).
    """
    rows = [to_V(z * b + b**q) for b in basis_F_q_n]
    return Matrix(F_q, rows)


def matrix_poly_eval_row_action(A, g_poly):
    """
    Evaluate polynomial g_poly at matrix A using Horner's rule.
    Row-action convention.
    """
    F_q = A.base_ring()
    n = A.nrows()
    I = identity_matrix(F_q, n)
    H = zero_matrix(F_q, n)

    for a in reversed(g_poly.list()):
        H = H * A
        if a != 0:
            H += F_q(a) * I

    return H


# ============================================================
# Factor data for m_{gamma_z}
# ============================================================

def m_gamma_factor_data(f_z, n, p, F_q_polynomials):
    """
    Compute factor data (polynomial itself, factorization & distinct factors and quotient_polynomials, i.e. m_gamma / factor_i) for
        m_gamma = f_z^(n/d) - 1
    using the p-part shortcut:
        if n/d = p^r * s with gcd(s, p) = 1, then
            f_z^(n/d) - 1 = (f_z^s - 1)^(p^r), where p is the characteristic of the field.
            It is then simpler to factorize f_z^s - 1 and adjust the multiplicities of the factors of m_gamma by multiplying with p^r.

    Theory:
    From Hsu and Nan (2011), it is known that the minimal polynomial of gamma_z is m_{gamma_z} = f_z^(n/deg(f_z)) - 1, 
    where f_z is the minimal polynomial of z over F_q.

    Returns:
        m_gamma        : full polynomial in F_q[t]
        reduced_fac    : factorization of f_z^s - 1
        full_fac       : factorization of m_gamma
        quotient_polys : [m_gamma / p_i] for distinct irreducible p_i
        distinct_facs  : [p_1, ..., p_l]
    """
    d = f_z.degree()
    e = ZZ(n // d)

    r = e.valuation(p)
    s = e // (p**r)

    reduced = f_z**s - F_q_polynomials(1)
    reduced_fac = list(reduced.factor())

    full_fac = [(pi, ai * (p**r)) for (pi, ai) in reduced_fac]

    m_gamma = F_q_polynomials(1)
    for pi, ei in full_fac:
        m_gamma *= pi**ei

    distinct_facs = [pi for (pi, _) in full_fac]
    quotient_polys = [m_gamma.quo_rem(pi)[0] for pi in distinct_facs]

    return m_gamma, reduced_fac, full_fac, quotient_polys, distinct_facs


# ============================================================
# Tau-generator testing strategies
# ============================================================

def precompute_tau_generator_test_data(
    z,
    orbit_z,
    F_q,
    n,
    lin_alg_data,
):
    """
    Precompute all fixed data needed to test whether x is a tau-generator for
    tau = gamma_z.

    Returns a dictionary containing:
        f_z, d, m_gamma, factorizations, quotient_polys, A, H_mats
    """
    q = F_q.order()
    p = F_q.characteristic()
    F_q_polynomials = PolynomialRing(F_q, 't')

    V, from_V, to_V, basis_F_q_n, zero_v = lin_alg_data

    f_z = minpoly_from_q_frobenius_orbit(orbit_z, F_q, var='t')
    d = len(orbit_z)

    m_gamma, reduced_fac, full_fac, quotient_polys, distinct_facs = \
        m_gamma_factor_data(f_z, n, p, F_q_polynomials)

    A = gamma_matrix_row_action(z, q, F_q, basis_F_q_n, to_V)
    H_mats = [matrix_poly_eval_row_action(A, g_poly) for g_poly in quotient_polys]

    return {
        'z': z,
        'orbit_z': orbit_z,
        'f_z': f_z,
        'd': d,
        'm_gamma': m_gamma,
        'reduced_factorization': reduced_fac,
        'full_factorization': full_fac,
        'distinct_irreducible_factors': distinct_facs,
        'quotient_polys': quotient_polys,
        'A': A,
        'H_mats': H_mats,
    }

def _passes_all_evaluation_tests(vx_rep, H_mats, zero_v):
    """
    Return True iff vx_rep * H != 0 for every H in H_mats.
    """
    return all(vx_rep * H != zero_v for H in H_mats)


def tau_generator_via_evaluation(
    point_pairs,
    H_mats,
    zero_v,
    F_q,
    F_q_n,
):
    """
    Deterministic exhaustive method on primitive projective points:
    test one representative x_rep of each primitive F_q-point.

    Since tau-generatorhood is F_q-scalar invariant, if x_rep fails then every
    lambda*x_rep fails as well.

    If a point passes, convert x_rep to an actual primitive element on that
    same point.
    """
    checked_points = 0

    for point_exp, x_rep, vx_rep in point_pairs:
        checked_points += 1

        if _passes_all_evaluation_tests(vx_rep, H_mats, zero_v):
            primitive_x, primitive_exp = primitive_multiple_on_same_point(
                F_q=F_q,
                F_q_n=F_q_n,
                point_exp=point_exp,
                x_rep=x_rep,
            )
            return primitive_x, checked_points, primitive_exp

    return None, checked_points, None


# ============================================================
# Primitive tau-generator search for one fixed z
# ============================================================

def find_primitive_tau_generator_for_z(
    z,
    orbit_z,
    F_q,
    F_q_n,
    n,
    lin_alg_data,
    point_pairs,
    return_data=False,
):
    """
    For fixed z in F_q_n with q-Frobenius orbit orbit_z, let tau = gamma_z.
    Find a primitive tau-generator, if one exists.

    Method: test one representative per primitive F_q-point by vx*H_i != 0
    """

    V, from_V, to_V, basis_F_q_n, zero_v = lin_alg_data

    precomp = precompute_tau_generator_test_data(
        z=z,
        orbit_z=orbit_z,
        F_q=F_q,
        n=n,
        lin_alg_data=lin_alg_data,
    )

    H_mats = precomp['H_mats']

    witness, checked_points, witness_exp = \
                tau_generator_via_evaluation(
                    point_pairs=point_pairs,
                    H_mats=H_mats,
                    zero_v=zero_v,
                    F_q=F_q,
                    F_q_n=F_q_n,
                )

    if not return_data:
        return witness

    data = dict(precomp)
    data['primitive_points_checked'] = checked_points
    data['witness_exponent'] = witness_exp

    return witness, data


# ============================================================
# Full Carlitz-extensivity test
# ============================================================

def is_carlitz_extensive(
    q,
    n,
    verbose=False,
    return_data=False,
    cache_primitive_candidates=True,
):
    """
    Test Carlitz-extensivity of F_{q^n} / F_q.

    Main ideas:
      - treat z = 0 separately
      - treat nonzero z via q-cyclotomic cosets mod q^n - 1
      - factor only one m_{gamma_z} per Frobenius orbit / minimal polynomial
      - test one representative per primitive F_q-point
      - optionally cache the primitive-point representatives
      - transport witnesses along each orbit by x |-> x^q

    Method: tests vx * H_i != 0 directly

    Returns:
        True / False

    If return_data=True, returns:
        (True/False, F_q, F_q_n, witnesses, diagnostics)
    """

    F_q, F_q_n = make_extension_field(q, n)
    q0 = F_q.order()

    lin_alg_data = prepare_linear_algebra(F_q, F_q_n)
    V, from_V, to_V, basis_F_q_n, zero_v = lin_alg_data

    point_pairs_cache = None
    if cache_primitive_candidates:
        point_pairs_cache = primitive_point_pair_list(F_q, F_q_n, to_V)

    witnesses = {}
    diagnostics = []

    def point_pairs_source():
        if point_pairs_cache is not None:
            return point_pairs_cache
        return primitive_point_pair_iter(F_q, F_q_n, to_V)

    # --------------------------------------------------------
    # Case z = 0
    # --------------------------------------------------------
    z0 = F_q_n(0)
    orbit0 = [z0]

    primitive_x0, data0 = find_primitive_tau_generator_for_z(
        z=z0,
        orbit_z=orbit0,
        F_q=F_q,
        F_q_n=F_q_n,
        n=n,
        lin_alg_data=lin_alg_data,
        point_pairs=point_pairs_source(),
        return_data=True,
    )

    diagnostics.append(data0)

    if primitive_x0 is None:
        if verbose:
            print("No primitive tau-generator exists for z = 0.")
        if return_data:
            return False, F_q, F_q_n, witnesses, diagnostics
        return False

    witnesses[z0] = primitive_x0

    if verbose:
        print(
            f"z = 0, deg(f_z) = {data0['d']}, "
            f"primitive points checked = {data0['primitive_points_checked']}"
        )

    # --------------------------------------------------------
    # Nonzero z: one representative per q-cyclotomic coset
    # --------------------------------------------------------
    printing_index = 0
    for coset_data in nonzero_q_cyclotomic_coset_data(F_q_n, q0):
        z_rep = coset_data['z0']
        orbit_z = coset_data['orbit_z']

        primitive_x_rep, data = find_primitive_tau_generator_for_z(
            z=z_rep,
            orbit_z=orbit_z,
            F_q=F_q,
            F_q_n=F_q_n,
            n=n,
            lin_alg_data=lin_alg_data,
            point_pairs=point_pairs_source(),
            return_data=True,
        )

        diagnostics.append(data)

        if primitive_x_rep is None:
            if verbose:
                print(
                    f"No primitive tau-generator exists for orbit representative "
                    f"z = {z_rep}."
                )
            if return_data:
                return False, F_q, F_q_n, witnesses, diagnostics
            return False

        # Transport witness along the Frobenius orbit:
        # if z_j = z_rep^(q^j), then x_j = x_rep^(q^j) works for z_j.
        xj = primitive_x_rep
        for zj in orbit_z:
            witnesses[zj] = xj
            xj = xj**q0
            
        printing_index += 1
        if verbose and printing_index % 30000 == 0:
            print(
                f"rep exp = {coset_data['rep_exp']}, "
                f"z = {z_rep}, deg(f_z) = {data['d']}, "
                f"primitive points checked = {data['primitive_points_checked']}"
            )

    if return_data:
        return True, F_q, F_q_n, witnesses, diagnostics
    return True

In [ ]:
import time

t0 = time.perf_counter()

ok, F_q, F_q_n, witnesses, diagnostics = is_carlitz_extensive(
    q=2,
    n=2,
    verbose=True,
    return_data=True,
    cache_primitive_candidates=True,
)

dt = time.perf_counter() - t0

print()
print("Result:", ok)
print(f"Elapsed time: {dt:.6f} seconds")
print("Number of diagnostics entries:", len(diagnostics))
print("Number of witnesses found:", len(witnesses))

z = 0, deg(f_z) = 1, primitive points checked = 1
No primitive tau-generator exists for orbit representative z = a.

Result: False
Elapsed time: 0.006783 seconds
Number of diagnostics entries: 3
Number of witnesses found: 2


In [ ]:
import time

t0 = time.perf_counter()

ok, F_q, F_q_n, witnesses, diagnostics = is_carlitz_extensive(
    q=2,
    n=18,
    verbose=True,
    return_data=True,
    cache_primitive_candidates=True,
)

dt = time.perf_counter() - t0

print()
print("Result:", ok)
print(f"Elapsed time: {dt:.6f} seconds")
print("Number of diagnostics entries:", len(diagnostics))
print("Number of witnesses found:", len(witnesses))

z = 0, deg(f_z) = 1, primitive points checked = 10

Result: True
Elapsed time: 118.194570 seconds
Number of diagnostics entries: 14602
Number of witnesses found: 262144


In [ ]:
import time

t0 = time.perf_counter()

ok, F_q, F_q_n, witnesses, diagnostics = is_carlitz_extensive(
    q=2,
    n=20,
    verbose=True,
    return_data=True,
    cache_primitive_candidates=True,
)

dt = time.perf_counter() - t0

print()
print("Result:", ok)
print(f"Elapsed time: {dt:.6f} seconds")
print("Number of diagnostics entries:", len(diagnostics))
print("Number of witnesses found:", len(witnesses))

z = 0, deg(f_z) = 1, primitive points checked = 6
rep exp = 78139, z = a^17 + a^15 + a^12 + a^9 + a^5, deg(f_z) = 20, primitive points checked = 2

Result: True
Elapsed time: 551.409839 seconds
Number of diagnostics entries: 52488
Number of witnesses found: 1048576


In [ ]:
import time

t0 = time.perf_counter()

ok, F_q, F_q_n, witnesses, diagnostics = is_carlitz_extensive(
    q=2,
    n=24,
    verbose=True,
    return_data=True,
    cache_primitive_candidates=True,
)

dt = time.perf_counter() - t0

print()
print("Result:", ok)
print(f"Elapsed time: {dt:.6f} seconds")
print("Number of diagnostics entries:", len(diagnostics))
print("Number of witnesses found:", len(witnesses))

z = 0, deg(f_z) = 1, primitive points checked = 5
rep exp = 60393, z = a^22 + a^20 + a^18 + a^17 + a^16 + a^15 + a^14 + a^11 + a^9 + a^4 + a^3 + a^2 + a + 1, deg(f_z) = 24, primitive points checked = 2
rep exp = 122057, z = a^23 + a^20 + a^13 + a^12 + a^11 + a^10 + a^7 + a^5, deg(f_z) = 24, primitive points checked = 5
rep exp = 185823, z = a^18 + a^14 + a^13 + a^11 + a^10 + a^5 + a + 1, deg(f_z) = 24, primitive points checked = 7
rep exp = 250527, z = a^23 + a^21 + a^20 + a^19 + a^18 + a^17 + a^15 + a^14 + a^10 + a^9 + a^8 + a^7 + a^3 + a^2 + 1, deg(f_z) = 24, primitive points checked = 2
rep exp = 320509, z = a^22 + a^21 + a^15 + a^14 + a^11 + a^10 + a^8 + a^7 + a^5 + a^4 + a^2 + a, deg(f_z) = 24, primitive points checked = 1
rep exp = 388991, z = a^23 + a^22 + a^21 + a^20 + a^19 + a^17 + a^14 + a^13 + a^12 + a^11 + a^9 + a^6 + a^5 + 1, deg(f_z) = 24, primitive points checked = 16
rep exp = 462709, z = a^22 + a^21 + a^19 + a^17 + a^16 + a^15 + a^13 + a^10 + a^9 + a^7 + a^4 + a^3 + a^

In [ ]:
import time

t0 = time.perf_counter()

ok, F_q, F_q_n, witnesses, diagnostics = is_carlitz_extensive(
    q=3,
    n=12,
    verbose=True,
    return_data=True,
    cache_primitive_candidates=True,
)

dt = time.perf_counter() - t0

print()
print("Result:", ok)
print(f"Elapsed time: {dt:.6f} seconds")
print("Number of diagnostics entries:", len(diagnostics))
print("Number of witnesses found:", len(witnesses))

z = 0, deg(f_z) = 1, primitive points checked = 5
rep exp = 66640, z = a^11 + 2*a^10 + 2*a^7 + 2*a^4 + a^2 + 2*a, deg(f_z) = 12, primitive points checked = 8

Result: True
Elapsed time: 284.710923 seconds
Number of diagnostics entries: 44368
Number of witnesses found: 531441


In [ ]:
import time

t0 = time.perf_counter()

ok, F_q, F_q_n, witnesses, diagnostics = is_carlitz_extensive(
    q=4,
    n=10,
    verbose=True,
    return_data=True,
    cache_primitive_candidates=True,
)

dt = time.perf_counter() - t0

print()
print("Result:", ok)
print(f"Elapsed time: {dt:.6f} seconds")
print("Number of diagnostics entries:", len(diagnostics))
print("Number of witnesses found:", len(witnesses))

z = 0, deg(f_z) = 1, primitive points checked = 1
rep exp = 43323, z = a^19 + a^18 + a^16 + a^13 + a^9 + a^8 + a^5 + a^4 + a^2, deg(f_z) = 10, primitive points checked = 4
rep exp = 103587, z = a^15 + a^14 + a^8 + a^7 + a^6 + a^5 + a^4 + a^3 + a, deg(f_z) = 10, primitive points checked = 3
rep exp = 211387, z = a^19 + a^18 + a^14 + a^12 + a^11 + a^10 + a^8 + a^7 + a^6 + a^5 + a^3 + a, deg(f_z) = 10, primitive points checked = 1

Result: True
Elapsed time: 1515.719882 seconds
Number of diagnostics entries: 104968
Number of witnesses found: 1048576


In [ ]:
import time

t0 = time.perf_counter()

ok, F_q, F_q_n, witnesses, diagnostics = is_carlitz_extensive(
    q=4,
    n=12,
    verbose=True,
    return_data=True,
    cache_primitive_candidates=True,
)

dt = time.perf_counter() - t0

print()
print("Result:", ok)
print(f"Elapsed time: {dt:.6f} seconds")
print("Number of diagnostics entries:", len(diagnostics))
print("Number of witnesses found:", len(witnesses))

z = 0, deg(f_z) = 1, primitive points checked = 1
rep exp = 40106, z = a^22 + a^20 + a^19 + a^18 + a^17 + a^15 + a^13 + a^11 + a^10 + a^9 + a^8 + a^6 + a^4 + a^3 + a, deg(f_z) = 12, primitive points checked = 1
rep exp = 80689, z = a^22 + a^21 + a^19 + a^17 + a^13 + a^10 + a^9 + a^8 + a^7 + a^2 + 1, deg(f_z) = 12, primitive points checked = 1
rep exp = 121491, z = a^23 + a^21 + a^19 + a^16 + a^12 + a^10 + a^7 + a^6 + a^5 + a + 1, deg(f_z) = 12, primitive points checked = 10
rep exp = 162965, z = a^15 + a^11 + a^8 + a^7 + a^6 + a^5 + a^4 + a^2, deg(f_z) = 12, primitive points checked = 3
rep exp = 205125, z = a^23 + a^22 + a^18 + a^17 + a^16 + a^15 + a^7 + a^6 + a^5, deg(f_z) = 12, primitive points checked = 2
rep exp = 247095, z = a^23 + a^22 + a^19 + a^15 + a^12 + a^5 + a^3 + a^2 + a, deg(f_z) = 12, primitive points checked = 1
rep exp = 293467, z = a^23 + a^22 + a^21 + a^19 + a^18 + a^17 + a^15 + a^13 + a^12 + a^11 + a^10 + a^7 + a^5 + a^4 + a^3 + a, deg(f_z) = 12, primitive points c

In [ ]:
import time

t0 = time.perf_counter()

ok, F_q, F_q_n, witnesses, diagnostics = is_carlitz_extensive(
    q=8,
    n=8,
    verbose=True,
    return_data=True,
    cache_primitive_candidates=True,
)

dt = time.perf_counter() - t0

print()
print("Result:", ok)
print(f"Elapsed time: {dt:.6f} seconds")
print("Number of diagnostics entries:", len(diagnostics))
print("Number of witnesses found:", len(witnesses))

z = 0, deg(f_z) = 1, primitive points checked = 1
rep exp = 34375, z = a^22 + a^18 + a^13 + a^9 + a^6 + a^5 + a^3 + 1, deg(f_z) = 8, primitive points checked = 1
rep exp = 68943, z = a^21 + a^20 + a^13 + a^11 + a^5 + a^3, deg(f_z) = 8, primitive points checked = 1
rep exp = 103726, z = a^23 + a^20 + a^19 + a^15 + a^14 + a^12 + a^9 + a^8 + a^7 + a^6 + a^5 + a^4 + 1, deg(f_z) = 8, primitive points checked = 1
rep exp = 138689, z = a^22 + a^21 + a^20 + a^19 + a^18 + a^17 + a^15 + a^14 + a^12 + a^7 + a, deg(f_z) = 8, primitive points checked = 1
rep exp = 173885, z = a^23 + a^20 + a^16 + a^14 + a^11 + a^9 + a^5 + a^4 + a^3 + 1, deg(f_z) = 8, primitive points checked = 1
rep exp = 209292, z = a^23 + a^18 + a^17 + a^16 + a^14 + a^13 + a^12 + a^11 + a^8 + a^7 + a^6 + a + 1, deg(f_z) = 8, primitive points checked = 1
rep exp = 244866, z = a^23 + a^22 + a^21 + a^19 + a^18 + a^16 + a^15 + a^14 + a^10 + a^9 + a^8 + a^5 + a^4, deg(f_z) = 8, primitive points checked = 1
rep exp = 284585, z = a^22 +